# 06 — Android Verification & Model Trade-off

Notebook này phục vụ **2 mục tiêu tách biệt**:

1. **Verification**: kiểm tra cùng một ảnh thì `best.pt` → `TFLite trên PC` → `TFLite trên Android` có cho kết quả gần nhau hay không. Đây là bước xác nhận pipeline Android không làm sai preprocessing / decode.
2. **Trade-off**: sau khi Android pipeline được xác nhận đúng, ghép:
   - Accuracy trên test set: Precision / Recall / mAP50 / mAP50-95
   - Android performance: latency / FPS / model size / RAM / thermal

Không dùng 10–20 ảnh verification để thay thế mAP trên toàn test set. Accuracy chính thức vẫn lấy từ full test set đã đánh giá trước đó.


## 1. Cấu hình đường dẫn

Mặc định:
- AI project: `D:\Project\TrafficSignAI`
- Android project: `D:\Project\AndroidProjects\TrafficSignApp`
- Dataset test: `VR-TSD-2/test/images`
- Android benchmark assets: `app/src/main/assets/benchmark_images`


In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"D:\Project\TrafficSignAI")
ANDROID_ROOT = Path(r"D:\Project\AndroidProjects\TrafficSignApp")

TEST_IMAGES = ROOT / "VR-TSD-2" / "test" / "images"
TEST_LABELS = ROOT / "VR-TSD-2" / "test" / "labels"

TRAINED_DIR = ROOT / "models" / "trained"
DEPLOY_DIR = ROOT / "models" / "deploy"

ANDROID_ASSET_IMAGES = (
    ANDROID_ROOT / "app" / "src" / "main" / "assets" / "benchmark_images"
)

VERIFY_DIR = ROOT / "android_verification"
VERIFY_DIR.mkdir(parents=True, exist_ok=True)

print("TEST_IMAGES:", TEST_IMAGES)
print("ANDROID_ASSET_IMAGES:", ANDROID_ASSET_IMAGES)
print("VERIFY_DIR:", VERIFY_DIR)


## 2. Chọn bộ ảnh cố định và copy sang Android

Android không tự có ảnh test. Cell này lấy 20 ảnh từ `test/images`, giữ nguyên filename và copy sang:

`app/src/main/assets/benchmark_images/`

Ta ưu tiên cả ảnh 1 object và nhiều object để verification đa dạng hơn. Label YOLO không cần copy sang Android.


In [ ]:
SEED = 42
NUM_VERIFY_IMAGES = 20

def label_count(label_path: Path) -> int:
    if not label_path.exists():
        return 0
    return len([
        x for x in label_path.read_text(encoding="utf-8").splitlines()
        if x.strip()
    ])

records = []
for image_path in sorted(TEST_IMAGES.iterdir()):
    if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
        continue

    label_path = TEST_LABELS / f"{image_path.stem}.txt"
    n = label_count(label_path)

    if n > 0:
        records.append({
            "image": image_path.name,
            "image_path": image_path,
            "gt_objects": n
        })

df_candidates = pd.DataFrame(records)

single = df_candidates[df_candidates["gt_objects"] == 1]
multi = df_candidates[df_candidates["gt_objects"] >= 2]

rng = np.random.default_rng(SEED)

n_multi = min(NUM_VERIFY_IMAGES // 2, len(multi))
n_single = NUM_VERIFY_IMAGES - n_multi

sel_multi = (
    multi.iloc[rng.choice(len(multi), size=n_multi, replace=False)]
    if n_multi > 0 else multi.iloc[:0]
)

sel_single = (
    single.iloc[rng.choice(len(single), size=min(n_single, len(single)), replace=False)]
    if len(single) > 0 else single.iloc[:0]
)

selected = pd.concat([sel_multi, sel_single], ignore_index=True)

if len(selected) < NUM_VERIFY_IMAGES:
    used = set(selected["image"])
    remaining = df_candidates[~df_candidates["image"].isin(used)]
    need = min(NUM_VERIFY_IMAGES - len(selected), len(remaining))
    extra = remaining.iloc[rng.choice(len(remaining), size=need, replace=False)]
    selected = pd.concat([selected, extra], ignore_index=True)

selected = selected.sample(frac=1, random_state=SEED).reset_index(drop=True)

ANDROID_ASSET_IMAGES.mkdir(parents=True, exist_ok=True)

for old in ANDROID_ASSET_IMAGES.iterdir():
    if old.is_file():
        old.unlink()

for row in selected.itertuples():
    shutil.copy2(row.image_path, ANDROID_ASSET_IMAGES / row.image)

manifest_path = VERIFY_DIR / "verification_manifest.csv"
selected[["image", "gt_objects"]].to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Đã copy {len(selected)} ảnh sang Android assets.")
print("Manifest:", manifest_path)
display(selected[["image", "gt_objects"]])


## 3. Verification model đang deploy

Bước verification ban đầu chỉ cần `yolo11n_640`, vì đây là model Android hiện tại.


In [ ]:
PT_MODEL = TRAINED_DIR / "yolo11n_640_best.pt"
TFLITE_MODEL = DEPLOY_DIR / "yolo11n_640.tflite"

print("PT exists:", PT_MODEL.exists(), PT_MODEL)
print("TFLite exists:", TFLITE_MODEL.exists(), TFLITE_MODEL)


## 4. Chạy đúng 20 ảnh bằng PT và TFLite trên PC

Mục tiêu là so tính nhất quán class / confidence / box, không phải benchmark tốc độ PC.


In [ ]:
from ultralytics import YOLO

pt_model = YOLO(str(PT_MODEL))
tflite_model = YOLO(str(TFLITE_MODEL), task="detect")

print("Đã load PT và TFLite.")


In [ ]:
def result_to_rows(result, image_name: str):
    detections = []

    if len(result.boxes) > 0:
        xyxy = result.boxes.xyxy.cpu().numpy()
        conf = result.boxes.conf.cpu().numpy()
        cls = result.boxes.cls.cpu().numpy().astype(int)

        for i in range(len(conf)):
            cid = int(cls[i])
            detections.append({
                "image": image_name,
                "detection_index": i,
                "class_id": cid,
                "label": result.names[cid],
                "confidence": float(conf[i]),
                "left": float(xyxy[i][0]),
                "top": float(xyxy[i][1]),
                "right": float(xyxy[i][2]),
                "bottom": float(xyxy[i][3]),
            })

    detections = sorted(
        detections,
        key=lambda x: x["confidence"],
        reverse=True
    )

    if detections:
        best = detections[0]
        summary = {
            "image": image_name,
            "detections": len(detections),
            "best_class_id": best["class_id"],
            "best_label": best["label"],
            "best_confidence": best["confidence"],
            "best_left": best["left"],
            "best_top": best["top"],
            "best_right": best["right"],
            "best_bottom": best["bottom"],
        }
    else:
        summary = {
            "image": image_name,
            "detections": 0,
            "best_class_id": -1,
            "best_label": "",
            "best_confidence": 0.0,
            "best_left": 0.0,
            "best_top": 0.0,
            "best_right": 0.0,
            "best_bottom": 0.0,
        }

    return summary, detections


def run_model_on_selected(model, prefix: str, device=None):
    summary_rows = []
    detection_rows = []

    for image_name in selected["image"]:
        image_path = TEST_IMAGES / image_name

        kwargs = dict(
            source=str(image_path),
            imgsz=640,
            conf=0.25,
            iou=0.45,
            verbose=False,
        )

        if device is not None:
            kwargs["device"] = device

        result = model.predict(**kwargs)[0]
        summary, detections = result_to_rows(result, image_name)

        summary_rows.append({
            "image": summary["image"],
            f"{prefix}_detections": summary["detections"],
            f"{prefix}_class_id": summary["best_class_id"],
            f"{prefix}_label": summary["best_label"],
            f"{prefix}_confidence": summary["best_confidence"],
            f"{prefix}_left": summary["best_left"],
            f"{prefix}_top": summary["best_top"],
            f"{prefix}_right": summary["best_right"],
            f"{prefix}_bottom": summary["best_bottom"],
        })

        for det in detections:
            det["source"] = prefix
            detection_rows.append(det)

    return pd.DataFrame(summary_rows), pd.DataFrame(detection_rows)


In [ ]:
pt_summary, pt_detections = run_model_on_selected(
    pt_model,
    prefix="pt",
    device=0
)

tflite_summary, tflite_detections = run_model_on_selected(
    tflite_model,
    prefix="tflite_pc"
)

pc_compare = (
    selected[["image", "gt_objects"]]
    .merge(pt_summary, on="image", how="left")
    .merge(tflite_summary, on="image", how="left")
)

pc_compare["pt_vs_tflite_class_match"] = (
    pc_compare["pt_class_id"] == pc_compare["tflite_pc_class_id"]
)

pc_compare["pt_vs_tflite_conf_diff"] = (
    pc_compare["pt_confidence"] - pc_compare["tflite_pc_confidence"]
).abs()

display(
    pc_compare[
        [
            "image",
            "gt_objects",
            "pt_label",
            "pt_confidence",
            "tflite_pc_label",
            "tflite_pc_confidence",
            "pt_vs_tflite_class_match",
            "pt_vs_tflite_conf_diff",
        ]
    ]
)


In [ ]:
pc_compare_path = VERIFY_DIR / "pc_pt_vs_tflite_summary.csv"
pc_detections_path = VERIFY_DIR / "pc_all_detections.csv"

pc_compare.to_csv(
    pc_compare_path,
    index=False,
    encoding="utf-8-sig"
)

pd.concat(
    [pt_detections, tflite_detections],
    ignore_index=True
).to_csv(
    pc_detections_path,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:")
print(pc_compare_path)
print(pc_detections_path)


## 5. Khi đã có CSV Android: so Android với TFLite PC

Sau khi app Android chạy đúng cùng `benchmark_images`, copy `android_summary_*.csv` về:

`D:\Project\TrafficSignAI\android_verification\`

Sau đó sửa filename bên dưới.


In [ ]:
ANDROID_SUMMARY_CSV = VERIFY_DIR / "android_summary_YYYYMMDD_HHMMSS.csv"

if ANDROID_SUMMARY_CSV.exists():
    android_df = pd.read_csv(ANDROID_SUMMARY_CSV)

    android_compare = android_df[
        [
            "image",
            "detections",
            "best_class_id",
            "best_label",
            "best_confidence",
            "best_left",
            "best_top",
            "best_right",
            "best_bottom",
            "inference_ms",
            "total_ms",
        ]
    ].rename(
        columns={
            "detections": "android_detections",
            "best_class_id": "android_class_id",
            "best_label": "android_label",
            "best_confidence": "android_confidence",
            "best_left": "android_left",
            "best_top": "android_top",
            "best_right": "android_right",
            "best_bottom": "android_bottom",
            "inference_ms": "android_inference_ms",
            "total_ms": "android_total_ms",
        }
    )

    verification = pc_compare.merge(
        android_compare,
        on="image",
        how="left"
    )

    verification["android_vs_tflite_class_match"] = (
        verification["android_class_id"]
        == verification["tflite_pc_class_id"]
    )

    verification["android_vs_tflite_conf_diff"] = (
        verification["android_confidence"]
        - verification["tflite_pc_confidence"]
    ).abs()

    display(
        verification[
            [
                "image",
                "tflite_pc_label",
                "tflite_pc_confidence",
                "android_label",
                "android_confidence",
                "android_vs_tflite_class_match",
                "android_vs_tflite_conf_diff",
                "android_inference_ms",
                "android_total_ms",
            ]
        ]
    )

    print(
        "Class match rate:",
        verification["android_vs_tflite_class_match"].mean()
    )
else:
    print(
        "Chưa có Android CSV. "
        "Sau khi chạy Android benchmark, copy CSV vào android_verification."
    )


# 6. Trade-off 4 model — accuracy đã có sẵn

Các giá trị dưới đây là test-set accuracy của **TFLite**, đã đo trước đó. Đây mới là accuracy dùng cho trade-off cuối cùng.


In [ ]:
model_accuracy = pd.DataFrame([
    {
        "model": "yolo11n_320",
        "input": 320,
        "precision": 0.7805,
        "recall": 0.6604,
        "mAP50": 0.7297,
        "mAP50_95": 0.5778,
        "tflite_size_mib": 10.166,
    },
    {
        "model": "yolo11n_640",
        "input": 640,
        "precision": 0.9132,
        "recall": 0.8322,
        "mAP50": 0.9203,
        "mAP50_95": 0.7496,
        "tflite_size_mib": 10.178,
    },
    {
        "model": "yolo11s_320",
        "input": 320,
        "precision": 0.9144,
        "recall": 0.8465,
        "mAP50": 0.8975,
        "mAP50_95": 0.7083,
        "tflite_size_mib": 36.206,
    },
    {
        "model": "yolo11s_640",
        "input": 640,
        "precision": 0.9352,
        "recall": 0.9421,
        "mAP50": 0.9704,
        "mAP50_95": 0.8064,
        "tflite_size_mib": 36.278,
    },
])

display(
    model_accuracy.sort_values(
        "mAP50_95",
        ascending=False
    )
)


## 7. Ghép benchmark Android của 4 model sau này

Sau khi app Android hỗ trợ `n320 / n640 / s320 / s640`, xuất CSV có cột `model`, `inference_ms`, `total_ms`.

Notebook sẽ tính median, p95 và FPS rồi ghép với accuracy.


In [ ]:
ANDROID_4MODEL_CSV = VERIFY_DIR / "android_4model_benchmark.csv"

if ANDROID_4MODEL_CSV.exists():
    android_perf = pd.read_csv(ANDROID_4MODEL_CSV)

    perf_summary = (
        android_perf
        .groupby("model")
        .agg(
            inference_median_ms=("inference_ms", "median"),
            inference_p95_ms=("inference_ms", lambda s: np.percentile(s, 95)),
            total_median_ms=("total_ms", "median"),
        )
        .reset_index()
    )

    perf_summary["estimated_fps"] = (
        1000.0 / perf_summary["total_median_ms"]
    )

    tradeoff = model_accuracy.merge(
        perf_summary,
        on="model",
        how="left"
    )

    display(
        tradeoff.sort_values(
            "mAP50_95",
            ascending=False
        )
    )
else:
    print(
        "Chưa có android_4model_benchmark.csv. "
        "File này sẽ có sau khi app hỗ trợ cả 4 model."
    )


## 8. Biểu đồ Pareto Accuracy ↔ Android Latency

Trục X thấp hơn tốt hơn, trục Y cao hơn tốt hơn. Model nằm gần góc trên-trái thường là lựa chọn cân bằng hơn.


In [ ]:
if "tradeoff" in globals() and tradeoff["total_median_ms"].notna().any():
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.scatter(
        tradeoff["total_median_ms"],
        tradeoff["mAP50_95"]
    )

    for _, row in tradeoff.iterrows():
        if pd.notna(row["total_median_ms"]):
            ax.annotate(
                row["model"],
                (row["total_median_ms"], row["mAP50_95"]),
                xytext=(5, 5),
                textcoords="offset points"
            )

    ax.set_xlabel("Android total latency median (ms) — lower is better")
    ax.set_ylabel("Test mAP50-95 — higher is better")
    ax.set_title("Android Model Trade-off: Accuracy vs Latency")
    ax.grid(True, alpha=0.3)

    plt.show()
else:
    print("Chưa có dữ liệu Android 4-model để vẽ Pareto.")


# Thứ tự đúng của project

```text
1. Train + đánh giá 4 model trên test set             ✅
2. Export 4 model sang TFLite                         ✅
3. Verify TFLite accuracy trên PC                     ✅
4. Deploy n640 lên Android                            ✅
5. Verify Android pipeline bằng cùng ảnh cố định      ← đang làm
6. Đưa cả 4 TFLite vào Android
7. Benchmark 4 model trên cùng thiết bị / cùng input
8. Ghép accuracy + latency + FPS + size + RAM + thermal
9. Phân tích trade-off / Pareto và chọn model cuối
```

Verification ở bước 5 không thay thế trade-off. Nó chỉ giúp tránh benchmark 4 model trên một pipeline Android đang xử lý sai.
